# 🤖 Lab 2: The Tech-Fluencer — Train an RNN to Generate LinkedIn Posts

> *"I woke up at 4 AM today to..."* — Your model writes the rest. 😎

---

**OBJECTIVE:** Take everything from the *Chaos to Numbers* worksheet (cleaning → tokenizing → padding → embedding → RNN) and **flip it**. Instead of *classifying* a message as spam, you'll teach an RNN to *generate* the next word, one at a time, until it has hallucinated a complete cringe LinkedIn post.

**THE SCENARIO:** A content marketing agency hired you. Their pitch: *"We use AI to write thought leadership at scale."* Your job: build the model that does it.

---

## 🧠 What's the same as the spam lab?

| Spam lab | This lab |
|---|---|
| `string.punctuation` cleaning | ✅ same |
| Keras `Tokenizer` | ✅ same |
| `pad_sequences` | ✅ same (with one twist — see Part 4) |
| `Sequential` + `Embedding` + RNN | ✅ same |
| `model.compile` / `model.fit` | ✅ same |

## 🆕 What's new? (only two real things)

1. **How we build training pairs.** Spam was *(message → 0/1)*. Here we need *(N words → next word)*. We call this **n-gram sequencing** — full hints provided.
2. **Generation.** After training, we sample words from the model in a loop with a temperature knob. Hints provided.

## 🎮 Self-grading

This notebook scores itself as you go. Every TODO has points (🟢 5 / 🟡 10 / 🔴 20), totalling **110 points** across 11 tasks. After each task, run the **`# 🎯 CHECK`** cell to validate your code, earn points, and see your live scoreboard. Climb the rank ladder:

- 🌱 **Curious Onlooker** (0–25%)
- 💼 **Networking Newbie** (25–50%)
- 🚀 **Aspiring Thought Leader** (50–70%)
- 📈 **Growth Hacker** (70–85%)
- 🦄 **Senior Disruptor** (85–99%)
- 👑 **Tech-Fluencer Royalty** (100%)

📌 Hints live in `<details>` blocks — open them only after you've genuinely tried.


## Part 0: Setup & Data 🟢 *(provided — no points, just run it)*

In [6]:
# Standard imports — same stack as the spam lab + numpy for sampling
import string
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))


TensorFlow version: 2.20.0
GPU available: True


### 🎮 Initialize the scoreboard

Run this cell once. It creates a `grader` object that tracks your points all the way through the lab. Don't re-run it later — it'll reset your score!

In [7]:
class TechFluencerGrader:
    """Self-grading system for the Tech-Fluencer lab."""

    RANKS = [
        (0.00, "🌱 Curious Onlooker",        "Just lurking. That's okay — for now."),
        (0.25, "💼 Networking Newbie",       "You've started showing up. The algorithm notices."),
        (0.50, "🚀 Aspiring Thought Leader", "Your DMs are open. Your hot takes are warm."),
        (0.70, "📈 Growth Hacker",           "You're posting daily. People are saving your posts."),
        (0.85, "🦄 Senior Disruptor",        "You give keynote talks. You write 'thread below.'"),
        (1.00, "👑 Tech-Fluencer Royalty",   "You speak only in synergies. The grind has paid off."),
    ]

    def __init__(self, total_possible=130):
        self.scores = {}     # task_id -> earned points
        self.max_points = {} # task_id -> max points
        self.total_possible = total_possible

    @property
    def total_earned(self):
        return sum(self.scores.values())

    def _bar(self, pct, width=30):
        filled = int(round(pct * width))
        return "█" * filled + "░" * (width - filled)

    def _rank(self):
        pct = self.total_earned / self.total_possible if self.total_possible else 0
        rank = self.RANKS[0]
        for threshold, name, blurb in self.RANKS:
            if pct >= threshold:
                rank = (threshold, name, blurb)
        return rank

    def check(self, task_id, passed, points, success_msg="", fail_msg="", hint=""):
        """Award points if passed. Idempotent — won't double-award on re-run."""
        self.max_points[task_id] = points
        already_passed = self.scores.get(task_id, 0) == points

        if passed:
            self.scores[task_id] = points
            badge = "🔁 Already complete" if already_passed else f"✅ +{points} points!"
            print(f"{badge}  [Task {task_id}]")
            if success_msg and not already_passed:
                print(f"   {success_msg}")
        else:
            if not already_passed:
                self.scores[task_id] = 0
            print(f"❌ Task {task_id} not passing yet. (0 / {points} points)")
            if fail_msg:
                print(f"   {fail_msg}")
            if hint:
                print(f"   💡 Hint: {hint}")

        self._show()

    def _show(self):
        pct = self.total_earned / self.total_possible if self.total_possible else 0
        _, rank_name, rank_blurb = self._rank()
        print()
        print(f"   Score:  {self.total_earned} / {self.total_possible}  [{self._bar(pct)}] {pct*100:.0f}%")
        print(f"   Rank:   {rank_name}")
        print(f"           {rank_blurb}")

    def scorecard(self):
        """Print full breakdown."""
        pct = self.total_earned / self.total_possible if self.total_possible else 0
        _, rank_name, rank_blurb = self._rank()
        print("=" * 56)
        print("🏆  FINAL SCORECARD")
        print("=" * 56)
        for tid in sorted(self.max_points.keys()):
            earned = self.scores.get(tid, 0)
            total  = self.max_points[tid]
            mark   = "✅" if earned == total else ("🟡" if earned > 0 else "❌")
            print(f"  {mark}  Task {tid:<8s}  {earned:>3d} / {total:<3d}")
        print("-" * 56)
        print(f"  TOTAL:               {self.total_earned:>3d} / {self.total_possible}")
        print(f"  [{self._bar(pct)}] {pct*100:.0f}%")
        print(f"  Rank: {rank_name}")
        print(f"        {rank_blurb}")
        print("=" * 56)


grader = TechFluencerGrader(total_possible=110)
print("🎮 Scoreboard initialized. Total points up for grabs: 110")
print("   Earn them by completing the TODOs in each Part.")


🎮 Scoreboard initialized. Total points up for grabs: 110
   Earn them by completing the TODOs in each Part.


### 📥 Get the dataset

We'll use the **LinkedIn Influencers' Data** from Kaggle:
🔗 https://www.kaggle.com/datasets/shreyasajal/linkedin-influencers-data

**Three ways to load it (try in order):**
- **Option A — Kaggle API in Colab.** Get `kaggle.json` from your Kaggle profile (Settings → API → Create New Token), then run the optional cell below.
- **Option B — Manual.** Download the CSV from the link, upload it to your Colab session as `influencers_data.csv`.
- **Option C — Built-in fallback.** If neither A nor B works, the next cell ships with 60 sample posts so you can keep moving.

In [8]:
# OPTION A (Colab only): uncomment to use Kaggle API
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d shreyasajal/linkedin-influencers-data
!unzip -o linkedin-influencers-data.zip


Saving influencers_data.csv to influencers_data.csv
cp: cannot stat 'kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/shreyasajal/linkedin-influencers-data
License(s): DbCL-1.0
100% 8.33M/8.33M [00:00<00:00, 40.3MB/s]

Archive:  linkedin-influencers-data.zip
  inflating: influencers_data.csv    


In [10]:
# Try real dataset; fall back to built-in sample if not found.
import os

df = None
for path in ["influencers_data.csv", "data/influencers_data.csv"]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Loaded real dataset from {path} — {len(df)} rows")
        break

if df is None:
    print("⚠️  Real dataset not found — using built-in sample of 60 tech-bro posts.")
    SAMPLE_POSTS = [
        "I woke up at 4 AM today to grind. While you were sleeping, I closed three deals. Hustle never sleeps. Are you in the 1 percent? Comment below.",
        "Just had the most insightful coffee chat with a Stanford dropout. Here are 7 lessons that will change your life forever. Save this post.",
        "Synergy is not a buzzword. It is a way of life. When my team aligned on our north star, revenue 10x. Let me explain how in 5 simple steps.",
        "Plot twist: I got rejected from Harvard. Best thing that ever happened to me. Failure is the new MBA. Tag someone who needs to hear this today.",
        "Unpopular opinion: Work life balance is a myth invented by quitters. I work 18 hours a day and I have never been happier. Hard work pays off.",
        "Today I fired my highest performer. Here is why. Culture eats strategy for breakfast. Sometimes the best hire is the one you let go.",
        "I scaled my startup from zero to one million dollars in 90 days using only LinkedIn. Here is the exact playbook. Thread below.",
        "Networking changed my life. Last week I cold messaged a billionaire and he replied in two minutes. Here is the framework I used.",
        "If you are not posting on LinkedIn every single day you are leaving money on the table. The algorithm rewards consistency. Trust me.",
        "I once was broke. Now I run three companies. The difference? Mindset. Your network is your net worth. Let me know your thoughts.",
        "ChatGPT will not replace you. A person using ChatGPT will replace you. Embrace AI or get left behind. The future is now.",
        "My morning routine: 5 AM wake up, cold shower, journal, gym, green smoothie, then ten cold emails before breakfast. What is yours?",
        "I asked my mentor what the secret to success was. His answer changed my life. Boring consistency beats inspired chaos every time.",
        "Quitting my six figure job to bet on myself was the scariest thing I ever did. Three years later I have zero regrets. Take the leap.",
        "Saying no is a superpower. I said no to 47 meetings last week. My productivity skyrocketed. Protect your calendar like it is gold.",
        "Just shipped our biggest feature ever with a team of three engineers in two weeks. Constraints breed creativity. Always have.",
        "Founder lesson number 38: Hire slow, fire fast. The wrong teammate will sink the ship faster than no teammate at all.",
        "I just turned down a 10 million dollar acquisition offer. Here is why. Not every exit is the right exit. Build for the long game.",
        "The best advice I ever got from a venture capitalist. Most companies do not die from starvation. They die from indigestion. Focus.",
        "When my cofounder quit on the day of our seed round, I cried in my car for an hour. Then I closed the round alone. Resilience is everything.",
        "Today my intern taught me more than my MBA ever did. Always be learning. The moment you stop is the moment you start losing.",
        "Three habits that 10x my productivity: time blocking, single tasking, and saying no to almost everything. Try them this week.",
        "I just got off a four hour call with a fortune 500 CEO. Here are the 12 things he said that completely changed my perspective on leadership.",
        "If you have to convince someone to hire you, you are the wrong hire. The best talent makes itself impossible to ignore. Be that person.",
        "Disruption is overrated. Execution is everything. Anyone can have an idea. Very few can ship. Be the rare ones.",
        "I lost 40 thousand dollars on my first startup. It was the best business school money could buy. Failure is just tuition. Pay it.",
        "The reason most people fail is not lack of skill. It is lack of speed. Move fast. Iterate. Ship before you are ready. Always.",
        "I read 52 books this year. Here are the 5 that actually mattered. Most books are noise. The right ones rewire your brain.",
        "My therapist told me something profound yesterday. Stop optimizing your life. Start living it. I have been thinking about it ever since.",
        "Just raised our series A. Here is what nobody tells you about fundraising. It is mostly rejection. Until suddenly it is not. Keep pitching.",
        "Hot take: most managers should be individual contributors. We promote people for being good at the job, not for being good at managing humans.",
        "I deleted twitter from my phone for 30 days. My focus came back. My anxiety dropped. My output doubled. Try it. You will thank me.",
        "Build in public. Sell in public. Fail in public. The reps you get from being seen are worth more than any course you will ever take.",
        "When you do not know what to do next, do the boring thing. Boring compounds. Exciting fizzles. The boring path is the rich path.",
        "I have hired over 200 people in my career. The single best predictor of success was not the resume. It was how they answered one question.",
        "Saying I do not know is a sign of strength, not weakness. The smartest people in the room are the ones most willing to admit ignorance.",
        "Cold email still works. I closed a six figure deal last week from a 4 line message. Stop overthinking it. Just send the email.",
        "Most overnight successes took ten years. Stop looking for shortcuts. Start showing up. Compounding is the eighth wonder of the world.",
        "If your meeting could have been an email, your job could have been automated. Be ruthless about your calendar. Ruthless.",
        "I run a 50 person company and I still answer every email myself. It keeps me honest. It keeps me close to the customer. Always will.",
        "The biggest lie in startup land is that you need to move to san francisco. I built mine from a small town in ohio. Geography is dead.",
        "Imposter syndrome is just a sign you care. Embrace it. The day you stop feeling it is the day you stop growing. Stay uncomfortable.",
        "Just had dinner with a unicorn founder. The thing that struck me most was how normal he was. The myth of the genius founder is dead.",
        "Pricing is positioning. Charge more. Watch what happens. Every time I have raised prices, I have attracted better customers. Every time.",
        "I do not believe in passion. I believe in interest plus practice. Passion is the result, not the cause. Get good at something boring first.",
        "The most underrated skill in business is writing clearly. If you cannot write a clear paragraph, you cannot lead a team. Period.",
        "Here is a framework I stole from a navy seal. Slow is smooth. Smooth is fast. When you feel rushed, slow down. You will move faster.",
        "Just shipped a side project on the weekend. Made my first dollar online. The feeling of internet money is unmatched. You can do this too.",
        "I have read every productivity book ever written. Here is the only thing that worked. Pick one thing. Do it before noon. Repeat tomorrow.",
        "If you are the smartest person in the room, you are in the wrong room. Surround yourself with people who make you feel slightly stupid.",
        "Burnout is real. I learned the hard way. Take the vacation. Take the nap. Take the walk. Output is a function of energy, not hours.",
        "The customer is not always right. But the customer is always the customer. Listen. Then decide. Do not just react. Lead.",
        "Most strategy decks die in the powerpoint. Strategy is what you do, not what you say. Show me your calendar and I will show you your strategy.",
        "I just turned 30 and here is everything I wish I knew at 20. Long thread incoming. The biggest lesson? Time is the only currency.",
        "Founders, your first hire matters more than your first investor. Get the people right and the money follows. Always in that order.",
        "Just got off stage at a conference of 5000 people. Five years ago I was too anxious to speak in meetings. Reps change everything. Reps.",
        "Stop calling yourself a perfectionist in interviews. It is the worst flex. Real perfectionists are slow. Speed wins. Be a finisher instead.",
        "The most generous thing you can do as a leader is be clear. Vague feedback is cruelty disguised as kindness. Say what you mean.",
        "I asked 100 founders what they would tell their younger selves. The most common answer? Talk to more customers. Earlier. Always.",
        "We just hit a million users. Here are the only three things that mattered. The rest was noise. Focus is a superpower in disguise.",
    ]
    df = pd.DataFrame({"content": SAMPLE_POSTS})
    print(f"✅ Sample dataset loaded — {len(df)} posts")

# Find the text column flexibly (real Kaggle CSV uses 'content')
text_col = next((c for c in ["content", "Content", "text", "post", "Post"] if c in df.columns), None)
if text_col is None:
    raise ValueError(f"No text column found. Available: {list(df.columns)}")

print(f"Using column: '{text_col}'")
df = df.dropna(subset=[text_col]).copy()
print(f"Final size: {len(df)} posts")
df[text_col].head(3)


✅ Loaded real dataset from influencers_data.csv — 34012 rows
Using column: 'content'
Final size: 31996 posts


/tmp/ipykernel_556/2232676085.py:7: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,content
0,Robert Lerman writes that achieving a healthy...
1,"National disability advocate Sara Hart Weir, ..."
3,Exploring in this months Talent Management & H...


## Part 1: Noise Removal & Normalization 🟡

This is **Part 1 from your worksheet** — exactly the same idea: lowercase + strip punctuation using `string.punctuation`. We just apply it to a whole column instead of one sentence.

In [11]:
# 🟡 TODO 1.1  (10 points)
# Write a clean_text(text) function that:
#   (a) Converts text to lowercase
#   (b) Removes all punctuation using string.punctuation (same as worksheet Part 1)
#   (c) Returns the result as a string
#
# This is the EXACT pattern from your worksheet. Look at Part 1 if you forget.

def clean_text(text):
  text=str(text).lower()
  translator=str.maketrans("","",string.punctuation)
  return text.translate(translator)


<details>
<summary>💡 Hint 1.1 — same as worksheet Part 1</summary>

```python
def clean_text(text):
    text = str(text).lower()
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)
```
</details>

In [12]:
# 🎯 CHECK 1.1 — run this to grade your clean_text function
try:
    out = clean_text("Ugh... The deliveries were DELAYED!! #annoyed")
    expected_words = {"ugh", "the", "deliveries", "were", "delayed", "annoyed"}
    actual_words = set(out.split())
    no_punct = all(ch not in out for ch in string.punctuation)
    is_lower = out == out.lower()
    passed = expected_words.issubset(actual_words) and no_punct and is_lower
    grader.check("1.1", passed, 10,
                 success_msg="Cleaning works! Lowercase + punctuation stripped.",
                 fail_msg=f"Got: {out!r}",
                 hint="Use text.lower() and str.maketrans('', '', string.punctuation).")
except Exception as e:
    grader.check("1.1", False, 10, fail_msg=f"Error: {e}",
                 hint="Make sure clean_text is defined and returns a string.")


✅ +10 points!  [Task 1.1]
   Cleaning works! Lowercase + punctuation stripped.

   Score:  10 / 110  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░] 9%
   Rank:   🌱 Curious Onlooker
           Just lurking. That's okay — for now.


In [15]:
# Apply cleaning to the whole dataset (provided)
df["clean"] = df[text_col].apply(clean_text)

# Filter out posts that are too short or too long for our model
word_counts = df["clean"].str.split().str.len()
df = df[(word_counts >= 20) & (word_counts <= 200)].reset_index(drop=True)

# For training speed, cap the corpus. Increase if you have GPU + patience.
MAX_POSTS = 500
if len(df) > MAX_POSTS:
    df = df.sample(MAX_POSTS, random_state=42).reset_index(drop=True)

print(f"After cleaning + filtering: {len(df)} posts")
print("\nExample cleaned post:")
print(df["clean"].iloc[0])


After cleaning + filtering: 500 posts

Example cleaned post:
what a fabulous way to start my tuesday—reading my dear friend  jamila robinson ’s joyous  instagram  post about her essay featured in this month’s  food and wine  magazine  im so proud to say that i’ve followed jamila’s journey from college intern at the  detroit free press  to today where she’s an editorial director for  atlantic 57  with a passion for all things culinary and refined but the most remarkable thing about jamila is her equal devotion to nurturing journalistic talent for people of color who want to write about  food   dining  and the  culinaryarts  through her work with  the james beard foundation  jamila has supported dozens of young journalists who might otherwise have not had the opportunity to pursue those fields three cheers for jamila—she already has my vote for “woman of the year”   womeninmedia   journalism   mentoring   journalistsofcolor   journalists   foodlovers   foodblogger   foodlove   foodfortho

## Part 2: Tokenizer & Vocabulary 🟢

This is **identical to Part 8 of your worksheet** — same `Tokenizer`, same `texts_to_sequences`. The only difference is what we feed it: LinkedIn posts instead of SMS messages.

🧠 **What `Tokenizer` does for you (free of charge):**
- Builds a vocabulary of the top `num_words` most common words.
- Assigns each word an integer ID (1, 2, 3, ...). ID 0 is reserved for padding.
- Replaces unknown words with an `<OOV>` token.
- Stores everything in `tokenizer.word_index` (a dict: word → int).

In [16]:
# 🟢 TODO 2.1  (5 points)
# Create and fit a Tokenizer on df["clean"], using the EXACT pattern from worksheet Part 8.
# - Use vocab_size = 5000
# - Use oov_token = "<OOV>"
# - Then call .fit_on_texts() on the cleaned posts.

VOCAB_SIZE = 5000
sentences=df["clean"].to_list()
tokenizer=Tokenizer(num_words=VOCAB_SIZE,oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)


<details>
<summary>💡 Hint 2.1 — copy-paste the worksheet pattern</summary>

```python
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
```
</details>

In [17]:
# 🎯 CHECK 2.1
try:
    has_word_index = hasattr(tokenizer, "word_index") and len(tokenizer.word_index) > 100
    has_oov = "<OOV>" in tokenizer.word_index
    grader.check("2.1", has_word_index and has_oov, 5,
                 success_msg=f"Tokenizer fit on {len(tokenizer.word_index):,} unique words. "
                             f"Top words: {list(tokenizer.word_index.items())[:5]}",
                 fail_msg="Tokenizer not fit correctly.",
                 hint="tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>'); "
                      "tokenizer.fit_on_texts(sentences)")
except NameError:
    grader.check("2.1", False, 5, fail_msg="`tokenizer` is not defined.",
                 hint="Did you actually create the Tokenizer object?")


✅ +5 points!  [Task 2.1]
   Tokenizer fit on 6,721 unique words. Top words: [('<OOV>', 1), ('the', 2), ('to', 3), ('and', 4), ('of', 5)]

   Score:  15 / 110  [████░░░░░░░░░░░░░░░░░░░░░░░░░░] 14%
   Rank:   🌱 Curious Onlooker
           Just lurking. That's okay — for now.


In [18]:
# 🟢 TODO 2.2  (5 points)
# Convert each post in `sentences` to a list of integer IDs using texts_to_sequences.
# This too is straight from worksheet Part 8.

# YOUR CODE HERE
input_sequences_per_post=tokenizer.texts_to_sequences(sentences)

# Sanity peek (uncomment after filling in)
# print("First 15 IDs of first post:", input_sequences_per_post[0][:15])
# print("That post in words:        ", sentences[0][:80], "...")


*italicised text*<details>
<summary>💡 Hint 2.2</summary>

```python
input_sequences_per_post = tokenizer.texts_to_sequences(sentences)
```
</details>

In [19]:
# 🎯 CHECK 2.2
try:
    is_list = isinstance(input_sequences_per_post, list)
    nonempty = len(input_sequences_per_post) == len(sentences)
    has_ints = isinstance(input_sequences_per_post[0][0], int)
    grader.check("2.2", is_list and nonempty and has_ints, 5,
                 success_msg=f"Got {len(input_sequences_per_post)} sequences of integer IDs.",
                 fail_msg="Output should be a list of lists of ints.",
                 hint="input_sequences_per_post = tokenizer.texts_to_sequences(sentences)")
except NameError:
    grader.check("2.2", False, 5, fail_msg="`input_sequences_per_post` is not defined.")


✅ +5 points!  [Task 2.2]
   Got 500 sequences of integer IDs.

   Score:  20 / 110  [█████░░░░░░░░░░░░░░░░░░░░░░░░░] 18%
   Rank:   🌱 Curious Onlooker
           Just lurking. That's okay — for now.


## Part 3: N-Gram Sequences 🔴 *(NEW — this is the key insight)*

In the spam lab, each training pair was *(whole message → 0 or 1)*. For text generation we need pairs of *(seen-so-far → next word)*. This is called **n-gram sequencing**.

### 🧠 The intuition

Imagine one cleaned post tokenizes to IDs `[10, 7, 22, 5, 91]` (= 5 words). We want to teach the model:

| Input (what it has seen) | Target (what comes next) |
|---|---|
| `[10]` | `7` |
| `[10, 7]` | `22` |
| `[10, 7, 22]` | `5` |
| `[10, 7, 22, 5]` | `91` |

So **one 5-word post produces 4 training examples**. We do this for every post, then concatenate. We'll keep the input + target glued together for now (last element = target) and split them after padding.

In [21]:
# 🔴 TODO 3.1  (20 points)  ← the meatiest task in the whole lab
# Build an `input_sequences` list. For each tokenized post in `input_sequences_per_post`,
# create n-gram sequences of length 2, 3, 4, ..., len(post).
#
# Example:
#   post = [10, 7, 22, 5, 91]
#   should add to input_sequences:
#     [10, 7]
#     [10, 7, 22]
#     [10, 7, 22, 5]
#     [10, 7, 22, 5, 91]
#
# Do this for EVERY post. Skip posts shorter than 2 tokens.

input_sequences=[]
for token in input_sequences_per_post:
  for i in range(len(token)):
    n_gram=token[:i+1]
    input_sequences.append(n_gram)



# YOUR CODE HERE  (use a nested for loop)


print(f"Generated {len(input_sequences):,} n-gram training sequences from {len(input_sequences_per_post)} posts.")
print(f"First 3 n-grams from post 0: {input_sequences[:3]}")


Generated 31,647 n-gram training sequences from 500 posts.
First 3 n-grams from post 0: [[27], [27, 6], [27, 6, 1679]]


<details>
<summary>💡 Hint 3.1 — the n-gram loop</summary>

```python
input_sequences = []
for token_list in input_sequences_per_post:
    for i in range(1, len(token_list)):
        n_gram = token_list[: i + 1]   # slice from start up to and including index i
        input_sequences.append(n_gram)
```

**Why does this work?** `i` ranges from 1 to `len(token_list) - 1`. At each step we take the slice ending at position `i`. The last element of the slice is the target; everything before it is the input. We'll separate them after padding in Part 4.
</details>

In [22]:
# 🎯 CHECK 3.1
try:
    is_list = isinstance(input_sequences, list)
    not_empty = len(input_sequences) > 100
    has_pair = any(len(s) == 2 for s in input_sequences)
    has_triple = any(len(s) == 3 for s in input_sequences)
    well_formed = all(isinstance(s, list) and all(isinstance(t, int) for t in s)
                      for s in input_sequences[:50])
    passed = is_list and not_empty and has_pair and has_triple and well_formed
    grader.check("3.1", passed, 20,
                 success_msg=f"Built {len(input_sequences):,} n-grams. "
                             f"Lengths range {min(len(s) for s in input_sequences)}–"
                             f"{max(len(s) for s in input_sequences)}.",
                 fail_msg="N-gram list looks off.",
                 hint="for token_list in input_sequences_per_post: "
                      "for i in range(1, len(token_list)): "
                      "input_sequences.append(token_list[:i+1])")
except NameError:
    grader.check("3.1", False, 20, fail_msg="`input_sequences` is not defined.")


✅ +20 points!  [Task 3.1]
   Built 31,647 n-grams. Lengths range 1–200.

   Score:  40 / 110  [███████████░░░░░░░░░░░░░░░░░░░] 36%
   Rank:   💼 Networking Newbie
           You've started showing up. The algorithm notices.


## Part 4: Padding & X / y Split 🟡

Same `pad_sequences` from your worksheet — but with **one critical change**:

### 🚨 `padding='pre'` (not `'post'` like in spam lab)

Why? Because the **target is the last element** of each n-gram. We want the most recent context glued to the right side, with zeros padding the left:

```
Spam lab (worksheet):  [10, 22, 5, 0, 0, 0]   ← padding='post'
This lab:              [0, 0, 0, 10, 22, 5]   ← padding='pre'
                                          ↑ target lives here
```

In [23]:
# 🟡 TODO 4.1  (10 points)
# 1. Find the length of the longest n-gram (we'll pad everything to this length).
# 2. Use pad_sequences with padding='pre' to convert input_sequences -> a numpy array.
#
# Hint: max(len(x) for x in input_sequences) gives you the max length.

# YOUR CODE HERE
max_seq_len = max(len(x) for x in input_sequences)
padded = pad_sequences(input_sequences ,max_seq_len , padding='pre' )

print(f"Max sequence length: {max_seq_len}")
print(f"Padded shape:        {padded.shape}")
print(f"Example padded row:  {padded[0]}")


Max sequence length: 200
Padded shape:        (31647, 200)
Example padded row:  [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0 27]


<details>
<summary>💡 Hint 4.1</summary>

```python
max_seq_len = max(len(x) for x in input_sequences)
padded = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))
```

The only difference from worksheet Part 8 is `padding='pre'` instead of `'post'`.
</details>

In [24]:
# 🎯 CHECK 4.1
try:
    correct_shape = padded.shape == (len(input_sequences), max_seq_len)
    is_pre_padded = padded[0].tolist().count(0) > 0
    grader.check("4.1", correct_shape and is_pre_padded, 10,
                 success_msg=f"Padded matrix shape {padded.shape}, pre-padded with zeros on the left.",
                 fail_msg=f"Padding shape: {padded.shape}",
                 hint="pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')")
except NameError:
    grader.check("4.1", False, 10, fail_msg="`padded` or `max_seq_len` not defined.")


✅ +10 points!  [Task 4.1]
   Padded matrix shape (31647, 200), pre-padded with zeros on the left.

   Score:  50 / 110  [██████████████░░░░░░░░░░░░░░░░] 45%
   Rank:   💼 Networking Newbie
           You've started showing up. The algorithm notices.


In [25]:
# 🟢 TODO 4.2  (5 points)
# Split each padded row:
#   X = everything except the last column        -> the input to the RNN
#   y = the last column only                     -> the target word ID we want to predict
#
# Hint: numpy slicing. padded[:, :-1] and padded[:, -1].

# YOUR CODE HERE
X = padded[:,:-1]
y = padded[:,-1]

print(f"X shape: {X.shape}   (input sequences, length {max_seq_len - 1})")
print(f"y shape: {y.shape}   (one target ID per row)")
print(f"Example: X[0] = {X[0]}")
print(f"         y[0] = {y[0]}  (the word ID we want the model to predict)")


X shape: (31647, 199)   (input sequences, length 199)
y shape: (31647,)   (one target ID per row)
Example: X[0] = [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
         y[0] = 27  (the word ID we want the model to predict)


<details>
<summary>💡 Hint 4.2</summary>

```python
X = padded[:, :-1]
y = padded[:, -1]
```
</details>

In [26]:
# 🎯 CHECK 4.2
try:
    correct_X = X.shape == (len(input_sequences), max_seq_len - 1)
    correct_y = y.shape == (len(input_sequences),)
    grader.check("4.2", correct_X and correct_y, 5,
                 success_msg=f"X is {X.shape}, y is {y.shape}. Ready to train.",
                 fail_msg=f"Got X={X.shape}, y={y.shape}",
                 hint="X = padded[:, :-1]; y = padded[:, -1]")
except NameError:
    grader.check("4.2", False, 5, fail_msg="X or y not defined.")


✅ +5 points!  [Task 4.2]
   X is (31647, 199), y is (31647,). Ready to train.

   Score:  55 / 110  [███████████████░░░░░░░░░░░░░░░] 50%
   Rank:   🚀 Aspiring Thought Leader
           Your DMs are open. Your hot takes are warm.


## Part 5: Build the RNN 🟡

Same `Sequential` API from your worksheet's spam model. Just two changes:

| Spam model (worksheet) | Generation model (here) | Why |
|---|---|---|
| `SimpleRNN(32)` | `LSTM(150)` | Long text needs better memory. Same Keras API — just swap the layer. |
| `Dense(1, activation='sigmoid')` | `Dense(VOCAB_SIZE, activation='softmax')` | Output is a *probability over every word in the vocab*, not a binary score. |

Everything else (the `Embedding` layer, the `Sequential` wrapper, `model.add()`) is identical.

In [27]:
# We'll need the actual vocab size that the tokenizer ended up using.
# tokenizer.word_index has every word it ever saw; +1 for the padding token (id 0).
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)
EMBED_DIM = 64
RNN_UNITS = 150

print(f"Real vocab size for the model: {VOCAB_SIZE_REAL}")
print(f"Input length (X.shape[1]):      {X.shape[1]}")


Real vocab size for the model: 5000
Input length (X.shape[1]):      199


In [28]:
# 🟡 TODO 5.1  (10 points)  — Build the model layer by layer.
# Use the EXACT pattern from worksheet Part 8 (Sequential + model.add).
#
# Architecture:
#   1. Embedding(input_dim=VOCAB_SIZE_REAL, output_dim=EMBED_DIM, input_length=X.shape[1])
#   2. LSTM(RNN_UNITS)
#   3. Dropout(0.2)            <-- new, prevents overfitting
#   4. Dense(VOCAB_SIZE_REAL, activation='softmax')

model = Sequential()

# YOUR CODE HERE — call model.add() four times
model.add(Embedding(input_dim=VOCAB_SIZE_REAL, output_dim=EMBED_DIM, input_length=X.shape[1]))
model.add(LSTM(RNN_UNITS))
model.add(Dropout(0.2))
model.add(Dense(VOCAB_SIZE_REAL, activation='softmax'))



model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

<details>
<summary>💡 Hint 5.1</summary>

```python
model = Sequential()
model.add(Embedding(input_dim=VOCAB_SIZE_REAL,
                    output_dim=EMBED_DIM,
                    input_length=X.shape[1]))
model.add(LSTM(RNN_UNITS))
model.add(Dropout(0.2))
model.add(Dense(VOCAB_SIZE_REAL, activation='softmax'))
```
</details>

In [29]:
# 🎯 CHECK 5.1
try:
    layer_types = [type(layer).__name__ for layer in model.layers]
    has_embedding = "Embedding" in layer_types
    has_recurrent = ("LSTM" in layer_types) or ("SimpleRNN" in layer_types) or ("GRU" in layer_types)
    has_dense = "Dense" in layer_types
    last_layer = model.layers[-1]
    last_units = getattr(last_layer, "units", None)
    correct_output = last_units == VOCAB_SIZE_REAL
    passed = has_embedding and has_recurrent and has_dense and correct_output
    grader.check("5.1", passed, 10,
                 success_msg=f"Model built: {' → '.join(layer_types)}. "
                             f"Output dim {last_units} = vocab size. ✓",
                 fail_msg=f"Layers: {layer_types}, last Dense units: {last_units}",
                 hint="Need: Embedding → LSTM → Dropout → Dense(VOCAB_SIZE_REAL, softmax)")
except NameError:
    grader.check("5.1", False, 10, fail_msg="`model` not defined.")


✅ +10 points!  [Task 5.1]
   Model built: Embedding → LSTM → Dropout → Dense. Output dim 5000 = vocab size. ✓

   Score:  65 / 110  [██████████████████░░░░░░░░░░░░] 59%
   Rank:   🚀 Aspiring Thought Leader
           Your DMs are open. Your hot takes are warm.


## Part 6: Compile & Train 🟡

Same `model.compile` + `model.fit` from your worksheet. Two tweaks:

| Spam model | Generation model | Why |
|---|---|---|
| `loss='binary_crossentropy'` | `loss='sparse_categorical_crossentropy'` | We're picking 1 of `VOCAB_SIZE` classes (not 2). "Sparse" means we keep `y` as integers instead of one-hot encoding it. |
| `metrics=['accuracy']` | `metrics=['accuracy']` | Same. (It'll look low — that's normal for language modeling. There's no "right" next word, just *plausible* ones.) |

In [30]:
from tensorflow._api.v2.config import optimizer
# 🟡 TODO 6.1  (10 points)
# Compile the model with:
#   - loss = 'sparse_categorical_crossentropy'
#   - optimizer = 'adam'
#   - metrics = ['accuracy']

# YOUR CODE HERE
model.compile(optimizer='adam' , loss='sparse_categorical_crossentropy' , metrics=['accuracy'])


<details>
<summary>💡 Hint 6.1</summary>

```python
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
```
</details>

In [31]:
# 🎯 CHECK 6.1
try:
    loss_name = model.loss if isinstance(model.loss, str) else model.loss.__class__.__name__.lower()
    is_sparse_cce = "sparse_categorical_crossentropy" in str(loss_name).lower()
    grader.check("6.1", is_sparse_cce, 10,
                 success_msg=f"Compiled with sparse_categorical_crossentropy + Adam.",
                 fail_msg=f"Got loss = {loss_name}",
                 hint="loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy']")
except Exception as e:
    grader.check("6.1", False, 10, fail_msg=f"Error checking compile: {e}")


✅ +10 points!  [Task 6.1]
   Compiled with sparse_categorical_crossentropy + Adam.

   Score:  75 / 110  [████████████████████░░░░░░░░░░] 68%
   Rank:   🚀 Aspiring Thought Leader
           Your DMs are open. Your hot takes are warm.


In [32]:
# 🟢 TODO 6.2  (5 points) — Train the model.
# Use model.fit on (X, y) for EPOCHS epochs with a reasonable batch size.
# Hint: epochs=20, batch_size=128, verbose=1
# This will take a few minutes on CPU; faster on GPU.

EPOCHS = 20
BATCH_SIZE = 128

# YOUR CODE HERE
history = model.fit(X,y,epochs=EPOCHS , batch_size= BATCH_SIZE)


Epoch 1/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.0512 - loss: 6.9710
Epoch 2/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.0660 - loss: 6.5675
Epoch 3/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.0770 - loss: 6.4660
Epoch 4/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.0938 - loss: 6.3547
Epoch 5/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.1004 - loss: 6.2420
Epoch 6/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.1056 - loss: 6.1305
Epoch 7/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.1104 - loss: 6.0052
Epoch 8/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.1188 - loss: 5.8881
Epoch 9/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.1230 - loss: 5.7588
Epoch 10/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.1280 - loss: 5.6322
Epoch 11/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.1305 - loss: 5.5406
Epoch 12/20
248/248 ━━━━━━━━━━━━━━━━━━━━

<details>
<summary>💡 Hint 6.2</summary>

```python
history = model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
```

📉 You'll see loss go down. Accuracy will look low (maybe 10–30%) — that's *expected* for language modeling, because at any position there are many valid next words.
</details>

In [40]:
# 🎯 CHECK 6.2
try:
    has_history = 'history' in dir() and hasattr(history, 'history')
    final_loss = history.history.get('loss', [None])[-1]
    trained = has_history and final_loss is not None and final_loss < 8.0
    grader.check("6.2", trained, 5,
                 success_msg=f"Model trained! Final loss: {final_loss:.4f}",
                 fail_msg="`history` not produced or loss looks wrong.",
                 hint="history = model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE)")
except Exception as e:
    grader.check("6.2", False, 5, fail_msg=f"Training didn't complete: {e}")


🔁 Already complete  [Task 6.2]

   Score:  110 / 110  [██████████████████████████████] 100%
   Rank:   👑 Tech-Fluencer Royalty
           You speak only in synergies. The grind has paid off.


## Part 7: Generate Tech-Bro Posts 🔴 *(NEW concept, full hints below)*

The fun part. Given a seed phrase, we:
1. Tokenize the seed (same `texts_to_sequences` as before).
2. Pad it to length `max_seq_len - 1`.
3. Predict the probability distribution over the next word.
4. Sample one word from that distribution.
5. Append it to the seed. Repeat.

### 🌡️ Temperature

Before sampling, we reshape the probabilities by temperature `T`:
- `T → 0`: greedy, picks the single most likely word every time. Boring, repetitive.
- `T = 1.0`: samples from the model's actual distribution. Balanced.
- `T > 1.0`: flattens the distribution. More random. More creative. More nonsense.

The math: `new_probs = softmax(log(probs) / T)`.

In [34]:
# Helper: fast lookup from word ID -> word string. (Build once, use many times.)
index_to_word = {idx: word for word, idx in tokenizer.word_index.items()}
print("Sample lookups:", {1: index_to_word.get(1), 2: index_to_word.get(2), 3: index_to_word.get(3)})


Sample lookups: {1: '<OOV>', 2: 'the', 3: 'to'}


In [35]:
# 🔴 TODO 7.1  (20 points) — Implement the generation loop.
# Fill in the body of the `for` loop.

def generate_text(seed_text, num_words, model, max_seq_len, temperature=1.0):
    # "\"\"Generate `num_words` more words after `seed_text`.\"\"\"
    seed_text = clean_text(seed_text)  # apply same preprocessing as training!

    for _ in range(num_words):
        # ---- (a) Tokenize the current seed_text ----
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # YOUR CODE HERE

        # ---- (b) Pre-pad to length max_seq_len - 1 ----
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        # YOUR CODE HERE

        # ---- (c) Predict the probability distribution over the next word ----
        probs = model.predict(token_list, verbose=0)[0]   # shape: (VOCAB_SIZE_REAL,)
        # YOUR CODE HERE
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs) / np.sum(np.exp(probs))

        # ---- (d) Apply temperature, then sample ----
        # See hint below. Use np.random.choice(len(probs), p=probs)
        # YOUR CODE HERE
        predicted_id = np.random.choice(len(probs), p=probs)

        # ---- (e) Convert ID back to a word and append ----
        output_word = index_to_word.get(predicted_id, "")
        if output_word == "" or output_word == "<OOV>":
            break  # stop if the model has nothing to say
        seed_text += " " + output_word
        # YOUR CODE HERE

        pass  # remove this once you've filled in the steps

    return seed_text


# Quick test (you'll see real output once your code is filled in)
print(generate_text("I woke up at 4 am today to", num_words=20, model=model,
                    max_seq_len=max_seq_len, temperature=1.0))


i woke up at 4 am today to help toward nothing if i have to caregivers podcast instead to “there staff for a year is today our list


<details>
<summary>💡 Hint 7.1 — full body of the loop</summary>

```python
def generate_text(seed_text, num_words, model, max_seq_len, temperature=1.0):
    seed_text = clean_text(seed_text)
    for _ in range(num_words):
        # (a) tokenize
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # (b) pad (same pre-padding as training)
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        # (c) predict probabilities
        probs = model.predict(token_list, verbose=0)[0]
        # (d) apply temperature and sample
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs) / np.sum(np.exp(probs))
        predicted_id = np.random.choice(len(probs), p=probs)
        # (e) append the chosen word
        output_word = index_to_word.get(predicted_id, "")
        if output_word == "" or output_word == "<OOV>":
            break
        seed_text += " " + output_word
    return seed_text
```

**Why `np.log(...) / T` then `softmax` again?** Dividing the *log-probabilities* by `T` and re-normalizing is the standard temperature trick. Larger `T` → distribution becomes flatter → more diversity.
</details>

In [36]:
# 🎯 CHECK 7.1
try:
    out = generate_text("I woke up at 4 am today to", num_words=10, model=model,
                        max_seq_len=max_seq_len, temperature=1.0)
    is_str = isinstance(out, str)
    grew = len(out.split()) > len("I woke up at 4 am today to".split())
    grader.check("7.1", is_str and grew, 20,
                 success_msg=f"Generation works! Sample output:\n   '{out}'",
                 fail_msg=f"Generated: {out!r} — looks empty or unchanged.",
                 hint="Check (a) tokenize -> (b) pad -> (c) predict -> (d) sample -> (e) append.")
except Exception as e:
    grader.check("7.1", False, 20, fail_msg=f"generate_text raised: {e}",
                 hint="Open the hint dropdown above for the full loop body.")


✅ +20 points!  [Task 7.1]
   Generation works! Sample output:
   'i woke up at 4 am today to put the financial minutes every keys to watch not my'

   Score:  100 / 110  [███████████████████████████░░░] 91%
   Rank:   🦄 Senior Disruptor
           You give keynote talks. You write 'thread below.'


In [37]:
# 🟡 TODO 7.2  (10 points) — Generate 3 posts at 3 different temperatures.
# Use the SAME seed for all three so you can compare.
# Required: at least one temperature < 1.0 and one > 1.0.

SEED = "I woke up at 4 am today to"

# YOUR CODE HERE — print 3 generations with different temperatures.
# E.g.: temperatures = [0.4, 1.0, 1.5]
for T in [0.4, 1.0, 1.5]:
    print(f"\n--- Temperature {T} ---")
    print(generate_text(SEED, num_words=40, model=model,
                        max_seq_len=max_seq_len, temperature=T))


--- Temperature 0.4 ---
i woke up at 4 am today to be the

--- Temperature 1.0 ---
i woke up at 4 am today to rule subjects on this and thats how to use the grudge s reasons in make something about analyzed agents them certain videos via the story to those who are negative is my message more doesnt my multimedia central founder and

--- Temperature 1.5 ---
i woke up at 4 am today to appreciate launched interviews produces a picture of deletefacebook today opportunities to patients today truth after surprisingly buying is encouragement lacava africans theres felt before let the ways to facts will plan podcasts heres your involves thinking of work are shameful


<details>
<summary>💡 Hint 7.2</summary>

```python
for T in [0.4, 1.0, 1.5]:
    print(f"\n--- Temperature {T} ---")
    print(generate_text(SEED, num_words=40, model=model,
                        max_seq_len=max_seq_len, temperature=T))
```

🔍 **What to notice:** at low T the output gets repetitive; at high T it gets wild and incoherent. Around T=0.8–1.0 is usually the cringe sweet spot.
</details>

In [38]:
# 🎯 CHECK 7.2 — relaxed; we just verify you tried multiple temperatures and got distinct results.
try:
    out_low  = generate_text(SEED, num_words=15, model=model, max_seq_len=max_seq_len, temperature=0.4)
    out_mid  = generate_text(SEED, num_words=15, model=model, max_seq_len=max_seq_len, temperature=1.0)
    out_high = generate_text(SEED, num_words=15, model=model, max_seq_len=max_seq_len, temperature=1.5)
    distinct = len({out_low, out_mid, out_high}) >= 2
    grader.check("7.2", distinct, 10,
                 success_msg="Different temperatures produce different vibes. Nice work.",
                 fail_msg="All temperatures produced the same text — temperature scaling may be off.",
                 hint="Check the np.log(probs)/T then softmax-renormalize step.")
except Exception as e:
    grader.check("7.2", False, 10, fail_msg=f"Sweep raised: {e}")


✅ +10 points!  [Task 7.2]
   Different temperatures produce different vibes. Nice work.

   Score:  110 / 110  [██████████████████████████████] 100%
   Rank:   👑 Tech-Fluencer Royalty
           You speak only in synergies. The grind has paid off.


## 🏆 Final Scorecard

Run the cell below to see your full breakdown and unlocked rank.

In [39]:
grader.scorecard()


🏆  FINAL SCORECARD
  ✅  Task 1.1        10 / 10 
  ✅  Task 2.1         5 / 5  
  ✅  Task 2.2         5 / 5  
  ✅  Task 3.1        20 / 20 
  ✅  Task 4.1        10 / 10 
  ✅  Task 4.2         5 / 5  
  ✅  Task 5.1        10 / 10 
  ✅  Task 6.1        10 / 10 
  ✅  Task 6.2         5 / 5  
  ✅  Task 7.1        20 / 20 
  ✅  Task 7.2        10 / 10 
--------------------------------------------------------
  TOTAL:               110 / 110
  [██████████████████████████████] 100%
  Rank: 👑 Tech-Fluencer Royalty
        You speak only in synergies. The grind has paid off.


## Part 8: Bonus Challenges 🔥 *(beyond 100% — for the show-offs)*

Each bonus you ship is worth pride points (and bragging rights). No automatic grading — call your teacher over to demo.

### 🥉 Easy bonuses
- **Try `SimpleRNN(150)` instead of `LSTM(150)`.** Same Keras API. Compare quality. Why is LSTM better?
- **Train for 50 epochs instead of 20.** Does the loss keep dropping or plateau? At what point does the model start memorizing the training set?

### 🥈 Medium bonuses
- **Top-k sampling.** Modify `generate_text` so it only samples from the top-k most likely tokens (e.g. k=10) — even at high temperature this stays coherent.
- **Add a validation split** to `model.fit(..., validation_split=0.1)`. Plot train vs val loss using `matplotlib`. When does it start overfitting?
- **Two RNN layers stacked.** `LSTM(150, return_sequences=True)` followed by another `LSTM(150)`. More capacity. Does it help?

### 🥇 Hard bonuses
- **Top-p (nucleus) sampling.** Sample from the smallest set of tokens whose cumulative probability ≥ p (say 0.9). Implement and compare to top-k.
- **Beam search.** Instead of sampling one path, keep the top-B most likely sequences at each step. Pick the best at the end.
- **Subword tokenizer.** Replace the Keras `Tokenizer` with the `bert-base-uncased` tokenizer from your worksheet. Does the model handle rare words better?

### 🏆 Extreme bonus
- **Build a tiny Transformer decoder** on this same data. Compare against the LSTM. Welcome to the modern era.

---

## 🎓 What you just built

You implemented every piece of a small language model: text cleaning, vocabulary construction, n-gram sequence batching, an embedding-RNN-softmax stack with teacher-forced training, and temperature-sampled generation. The same conceptual stack — at much larger scale and with attention instead of recurrence — is what powers GPT-class models.

> Ship the post. Comment "🚀" below if this changed your life.
